In [2]:
# Task 2: Bigram Language Model using Gutenberg Corpus

import nltk
from nltk.corpus import gutenberg
from nltk.util import ngrams
from collections import Counter
import math
import pandas as pd

nltk.download('gutenberg')
nltk.download('punkt')

print("Resources loaded successfully!")

Resources loaded successfully!


[nltk_data] Downloading package gutenberg to /home/parth/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /home/parth/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [3]:
# (a) Load a suitable English text from Gutenberg
raw_words = gutenberg.words('austen-emma.txt')
print(f"Total raw words in Austen's Emma: {len(raw_words)}")

Total raw words in Austen's Emma: 192427


In [4]:
# (b) Tokenize and normalize the corpus
# Preprocessing: convert to lowercase and keep alphabetic words
tokens = [w.lower() for w in raw_words if w.isalpha()][:10000]
vocab = set(tokens)
vocab_size = len(vocab)

print(f"Total tokens extracted: {len(tokens)}")
print(f"Vocabulary size (V): {vocab_size}")
print(f"Sample tokens: {tokens[:15]}")

Total tokens extracted: 10000
Vocabulary size (V): 1762
Sample tokens: ['emma', 'by', 'jane', 'austen', 'volume', 'i', 'chapter', 'i', 'emma', 'woodhouse', 'handsome', 'clever', 'and', 'rich', 'with']


In [5]:
# (c) Construct unigram and bigram frequency counts
unigrams = tokens
bigrams = list(ngrams(tokens, 2))

unigram_counts = Counter(unigrams)
bigram_counts = Counter(bigrams)

print(f"Total Bigrams: {len(bigrams)}")
print(f"Unique Bigrams: {len(bigram_counts)}")
print("\nTop 5 Bigrams:", bigram_counts.most_common(5))

Total Bigrams: 9999
Unique Bigrams: 7087

Top 5 Bigrams: [(('of', 'the'), 45), (('to', 'be'), 35), (('she', 'had'), 31), (('it', 'was'), 30), (('in', 'the'), 28)]


In [6]:
# (d) Implement add-one (Laplace) smoothing
# Formula: P(w2 | w1) = (Count(w1, w2) + 1) / (Count(w1) + V)
def bigram_prob_add1(w1, w2):
    c_w1_w2 = bigram_counts.get((w1, w2), 0)
    c_w1 = unigram_counts.get(w1, 0)
    return (c_w1_w2 + 1) / (c_w1 + vocab_size)

print("Add-one smoothing function defined successfully!")

Add-one smoothing function defined successfully!


In [7]:
# (e) Calculate scores for at least three test sentences
test_sentences = [
    "she was a very quiet girl",
    "the young man was happy",
    "space rocket flying computer algorithm"
]

results = []
for s in test_sentences:
    words = [w.lower() for w in s.split() if w.isalpha()]
    log_prob = 0.0
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i+1]
        prob = bigram_prob_add1(w1, w2)
        log_prob += math.log(prob)
    
    results.append({
        "Test Sentence": s,
        "Log-Probability": round(log_prob, 4),
        "Estimated Probability": f"{math.exp(log_prob):.2e}"
    })

scores_df = pd.DataFrame(results)
display(scores_df)

,Test Sentence,Log-Probability,Estimated Probability
0,she was a very quiet girl,-28.1567,5.91e-13
1,the young man was happy,-26.5915,2.83e-12
2,space rocket flying computer algorithm,-29.8968,1.04e-13


## Observations

### (f) Importance of Smoothing in Language Modeling

1. **Zero Probability Problem:** In standard maximum likelihood estimation (MLE), if any word pair (bigram) in a test sentence never appeared in the training corpus, its estimated count is $0$, leading to $P(w_i \mid w_{i-1}) = 0$.
2. **Impact on Sentence Probability:** Because a sentence's total probability is calculated as the product of its individual bigram probabilities ($P(S) = \prod P(w_i \mid w_{i-1})$), a single zero causes the entire sentence probability to collapse to zero ($0$).
3. **Add-One (Laplace) Smoothing Solution:** By adding $1$ to the numerator and $V$ (vocabulary size) to the denominator, add-one smoothing redistributes probability mass, assigning a small non-zero probability to unseen pairs.
4. **Scoring Behavior:** Natural English sentences that align with Austen's vocabulary and phrasing (e.g., *"the young man was happy"*) receive substantially higher probabilities than out-of-domain modern phrases (e.g., *"space rocket flying computer algorithm"*).